# Join pieces into a square (tiny press-to-join demo)

The smallest version of the "press and the thing joins" interaction: **four loose bars** scattered on
a canvas. Press each one and it **snaps into place**, building a square frame. No model, no GPU - it
isolates the assembly mechanic so you can feel it.

- Press a bar -> it flies to its slot in the square.
- The faint outline shows the target square.
- 4 presses -> square complete. **Reset** to scatter them again.

In [ ]:
!pip install -q gradio pillow

In [ ]:
from PIL import Image, ImageDraw
import gradio as gr

# Each piece: its final (home) rectangle in the square, and where it sits while loose (scatter center)
PIECES = {
    "top":    {"home": (100, 100, 300, 120), "scatter": (60, 30)},
    "right":  {"home": (280, 100, 300, 300), "scatter": (370, 60)},
    "bottom": {"home": (100, 280, 300, 300), "scatter": (340, 370)},
    "left":   {"home": (100, 100, 120, 300), "scatter": (30, 340)},
}

def scatter_rect(p):
    x0, y0, x1, y1 = p["home"]
    hcx, hcy = (x0+x1)/2, (y0+y1)/2
    sx, sy = p["scatter"]
    ox, oy = sx-hcx, sy-hcy
    return (x0+ox, y0+oy, x1+ox, y1+oy)

def render(placed):
    img = Image.new("RGB", (400, 400), (245, 245, 245))
    d = ImageDraw.Draw(img)
    d.rectangle((100, 100, 300, 300), outline=(205, 205, 205), width=2)  # target square
    for name, p in PIECES.items():
        if name in placed:
            d.rectangle(p["home"], fill=(70, 180, 110))                  # snapped (green)
        else:
            d.rectangle(scatter_rect(p), fill=(150, 150, 160))           # loose (grey)
    return img

def on_click(placed, evt: gr.SelectData):
    placed = set(placed or [])
    x, y = evt.index
    best, bd = None, 1e18
    for name, p in PIECES.items():
        if name in placed:
            continue
        cx, cy = p["scatter"]
        dd = (cx-x)**2 + (cy-y)**2
        if dd < bd:
            bd, best = dd, name
    if best:
        placed.add(best)
    done = len(placed) == 4
    msg = str(len(placed)) + "/4 joined" + ("   -   square complete!" if done else "   (press a loose bar)")
    return render(placed), list(placed), msg

def reset():
    return render(set()), [], "0/4 joined   (press a loose bar)"

with gr.Blocks(title="Join into a square") as demo:
    gr.Markdown("## Press each bar to join it into a square")
    placed_state = gr.State([])
    label = gr.Markdown("0/4 joined   (press a loose bar)")
    canvas = gr.Image(value=render(set()), type="pil", label="Press a bar", interactive=False)
    rst = gr.Button("Reset")
    canvas.select(on_click, [placed_state], [canvas, placed_state, label])
    rst.click(reset, None, [canvas, placed_state, label])

demo.launch(share=True)

## Notes
- This is the same **press-to-join** idea as the main demo, reduced to 4 synthetic pieces so the
  assembly logic is obvious. Selection = nearest loose bar to your click; placement = snap to its slot.
- To make it a real manual, swap the synthetic bars for detected parts in a photo and the home slots
  for model-predicted target positions. The interaction code stays the same.